In [ ]:
import sys
sys.path.append('./experiments/llava') 

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch,torch.nn as nn
import os
from _utils import construct_train_test_ds, prepare_hal_train_test_ds, prepare_pair_data_loader, prepare_hal_train_ds
from tqdm import tqdm
import math
import time
from wrapper import Wrapper
from copy import deepcopy
from _utils import load_flow_model

model_name = "llava"

model_path = "/gz-data/models/llava-v1.5-7b" # your_path
image_path = "/gz-data/datasets/coco/train2014" # your_path

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

num_epochs = 25
token_pos = "last"
min_lr_scale = 0.7
num_warmup_steps = 100
layers = [15]
layer = layers[0]
k = 20
alpha = 2.0
ds_name = "rfi"

# Flow Matching Model Path
save_nn_name = f"Flow_{ds_name}_{model_name}_epoch{num_epochs}" # save neural network for flow

res_dir = f"{model_name}_hal_results"
if not os.path.exists(res_dir):
    os.makedirs(res_dir)
    
save_model_path = os.path.join(res_dir, save_nn_name) + f"_{layer}.pth" 


## Extract Data For FLOW TRAINING

In [2]:
# Prepare LVLM Hidden States data for Training FLOW MODEL

dataset = load_dataset("json", data_files="./experiments/data/correct_hal_pairs" + ".jsonl", split="train")


train_test_split = dataset.train_test_split(test_size=0.2)
train_ds = train_test_split["train"]
test_ds = train_test_split["test"]
print(f"Train size: {len(train_ds)}, Test size: {len(test_ds)}")

# load model and tokenizer
if "llava" in model_name:
    from llava.model.builder import load_pretrained_model
    from llava.mm_utils import get_model_name_from_path
    model_path = os.path.expanduser(model_path)
    model_base = None
    tokenizer, model, image_processor, context_len = load_pretrained_model(model_path, model_base, get_model_name_from_path(model_path), device=device)

model.eval()
hid_dim = model.config.hidden_size

dataset = construct_train_test_ds(
    train_ds,
    test_ds, 
    model_name, 
    model, 
    tokenizer,
    image_path,
    image_processor,  
    layers, 
    token_pos, 
    device=device 
)

# save dataset
os.makedirs(ds_name, exist_ok=True)
dataset.save_to_disk(os.path.join(ds_name, model_name + f"_{layer}"))

You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.


Train size: 80, Test size: 20


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 20/20 [00:12<00:00,  1.62it/s]


Saving the dataset (0/1 shards):   0%|          | 0/80 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/20 [00:00<?, ? examples/s]

## Train Flow Model

In [7]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"


ds_path = os.path.join(ds_name, model_name + f"_{layer}")

train_ds, val_ds = prepare_hal_train_ds(tokenizer, ds_path, model_name, device, layers)

flow_model = load_flow_model(hid_dim, device = device)

train_loader = prepare_pair_data_loader(train_ds, layers, ds_type="train")
val_loader = prepare_pair_data_loader(val_ds, layers, ds_type="test")

optimizer = torch.optim.AdamW(flow_model.parameters(), lr=1e-4)

num_training_steps = len(train_loader) * num_epochs


def cosine_schedule_with_warmup(current_step: int):
    if current_step < num_warmup_steps:
        return float(current_step) / float(max(1, num_warmup_steps))
    progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
    return max(min_lr_scale, cosine_decay)
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=cosine_schedule_with_warmup)

start_time = time.time()
train_losses = []
for epoch in range(num_epochs):
    # Training phase
    flow_model.train()
    train_loss = 0
    train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] - Training")
    for example in train_bar:
        y_win = example[f"y_win_layer{layer}"]
        y_lose = example[f"y_lose_layer{layer}"]
        y_win, y_lose = y_win.to(device), y_lose.to(device)
            
        loss = flow_model(y_win, y_lose, return_loss_breakdown = False)  
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()
        train_loss += loss.item()
        train_bar.set_postfix(train_loss=loss.item())
    train_loss /= len(train_loader)
    train_losses.append(train_loss)


    # validation phase
    if val_loader is None:
        continue
    flow_model.eval()
    val_loss = 0
    val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{num_epochs}] - Validation")
    with torch.no_grad():
        for example in val_bar:
            y_win = example[f"y_win_layer{layer}"]
            y_lose = example[f"y_lose_layer{layer}"]
            y_win, y_lose = y_win.to(device), y_lose.to(device)
            
            loss = flow_model(y_win, y_lose, return_loss_breakdown = False)  
            val_loss += loss.item()
            val_bar.set_postfix(val_loss=loss.item())
    
    val_loss /= len(val_loader)
    
    
end_time = time.time()
print(f"Training time: {end_time - start_time}")

# save model
torch.save(flow_model.state_dict(), save_model_path)


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

Epoch [25/25] - Validation: 100%|██████████| 1/1 [00:00<00:00, 205.57it/s, val_loss=0.15]


Training time: 1.8738703727722168


# Inference

In [ ]:
from llava.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN, DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN
from llava.conversation import conv_templates, 
from llava.mm_utils import tokenizer_image_token, process_images
from PIL import Image
import requests
from io import BytesIO



case_image_path = "./figs"
case_image_file = "case_albert.png"


def load_image(image_file, prefix_path):
    if image_file.startswith("http") or image_file.startswith("https"):
        response = requests.get(image_file)
        image = Image.open(BytesIO(response.content)).convert("RGB")
    else:
        image = Image.open(prefix_path + "/" + image_file).convert("RGB")
    return [image]


flow_model = load_flow_model(hid_dim, device, save_model_path)
flow_model.eval()
flow = []
flow.append(flow_model)

wrapper = Wrapper

ds_path = os.path.join(ds_name, model_name + f"_{layer}")

train_ds, _ = prepare_hal_train_test_ds(tokenizer, ds_path, model_name, image_path, model.config, device, layers, image_processor)

hs_mat = torch.cat([train_ds[i][f"y_win_layer{layer}"] for i in range(len(train_ds))], dim=0)
_, _, v = torch.svd(hs_mat)



qs = "What is the man's primary job?"
if model.config.mm_use_im_start_end:
    qs = DEFAULT_IM_START_TOKEN + DEFAULT_IMAGE_TOKEN + DEFAULT_IM_END_TOKEN + '\n' + qs
else:
    qs = DEFAULT_IMAGE_TOKEN + '\n' + qs
conv_mode = "llava_v1"
conv = conv_templates[conv_mode].copy()
conv.append_message(conv.roles[0], qs)
conv.append_message(conv.roles[1], None)
prompt = conv.get_prompt()
input_ids = tokenizer_image_token(prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt').unsqueeze(0).cuda(0)
images = load_image(case_image_file, case_image_path)
images_tensor = process_images(
    images,
    image_processor,
    model.config
).to(model.device, dtype=torch.float16)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [4]:
# Generate with original model

outputs = model.generate(
    input_ids,
    images=images_tensor,
    max_new_tokens=64,
    output_hidden_states=True,
    return_dict_in_generate=True,
)

output_ids = outputs.sequences
org_result = tokenizer.batch_decode(output_ids[:, :], skip_special_tokens=True)[0]


# Generate with RFI

with torch.no_grad():
    outputs = model(
        input_ids,
        images= images_tensor,
        output_hidden_states=True
    )

original_layers = []
for idx, layer in enumerate(layers):
    hs = outputs.hidden_states[layer][:, -1, :]
    hs_flow = flow[idx].sample(hidden_states=hs)
    original_layers.append(deepcopy(model.model.layers[layer]))
    model.model.layers[layer] = wrapper(model.model.layers[layer], hs_flow[0], v.to(device), k=k, alpha=alpha)

model.eval()
with torch.inference_mode():
    outputs = model.generate(
        input_ids, 
        images= images_tensor,
        max_new_tokens=64,
        return_dict_in_generate=True,
    )
    
output_ids = outputs.sequences
input_token_len = input_ids.shape[1]
result = tokenizer.batch_decode(output_ids[:, :], skip_special_tokens=True)[0]

for idx, layer in enumerate(layers):
    model.model.layers[layer] = original_layers[idx]



print("-----------Org-----------")
print(org_result)
print("-----------Flow----------")
print(result)

-----------Org-----------
The man's primary job is that of a musician, as he is holding a guitar and appears to be playing it.
-----------Flow----------
The man's primary job is that of a scientist, specifically a physicist.
